## Homework classification module

In [8]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv')

In [3]:
df

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1
...,...,...,...,...,...,...,...,...,...
1457,referral,manufacturing,1,NaN,self_employed,north_america,4,0.53,1
1458,referral,technology,3,65259.0,student,europe,2,0.24,1
1459,paid_ads,technology,1,45688.0,student,north_america,3,0.02,1
1460,referral,NaN,5,71016.0,self_employed,north_america,0,0.25,1


In [6]:
df.dtypes

lead_source                  object
industry                     object
number_of_courses_viewed      int64
annual_income               float64
employment_status            object
location                     object
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [5]:
df.isna().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [9]:
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

target_col = "converted"
assert target_col in df.columns, f"Coluna alvo '{target_col}' não encontrada. Colunas: {df.columns.tolist()}"

# Identificar tipos
cat_cols_all = df.select_dtypes(include=["object"]).columns.tolist()
num_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()

# Imputação de faltantes conforme enunciado
if len(cat_cols_all):
    df[cat_cols_all] = df[cat_cols_all].fillna("NA")
if len(num_cols_all):
    df[num_cols_all] = df[num_cols_all].fillna(0.0)

# Preparar y binário (0/1)
y_raw = df[target_col]
if y_raw.dtype == "O":
    y_lower = y_raw.astype(str).str.lower()
    if set(y_lower.unique()) <= {"yes", "no"}:
        y = y_lower.map({"yes": 1, "no": 0}).astype(int)
    elif set(y_lower.unique()) <= {"true", "false"}:
        y = y_lower.map({"true": 1, "false": 0}).astype(int)
    elif set(y_lower.unique()) <= {"1", "0"}:
        y = y_lower.map({"1": 1, "0": 0}).astype(int)
    else:
        y = pd.factorize(y_lower)[0]
else:
    y = (y_raw > 0).astype(int)

X = df.drop(columns=[target_col])

print("X shape:", X.shape, "| y distribution:", y.value_counts(normalize=True).round(3).to_dict())

X shape: (1462, 8) | y distribution: {1: 0.619, 0: 0.381}


Q1

In [10]:
df['industry'].mode()

0    retail
Name: industry, dtype: object

In [11]:
# === 3) Q1 — Moda da coluna 'industry' ===
q1_mode = None
if "industry" in df.columns:
    m = df["industry"].mode(dropna=False)
    q1_mode = m.iloc[0] if len(m) else None
else:
    q1_mode = None

choices_q1 = ["NA", "technology", "healthcare", "retail"]
q1_choice = None
if q1_mode is not None:
    cm = str(q1_mode).strip().lower()
    for ch in choices_q1:
        if cm == ch.lower():
            q1_choice = ch
            break
    if q1_choice is None and cm in {"", "na", "none"}:
        q1_choice = "NA"

print("Q1) moda(industry) =", q1_mode, "| opção:", q1_choice)

Q1) moda(industry) = retail | opção: retail


Q2

In [12]:
num_df = X.select_dtypes(include=[np.number])
corr = num_df.corr() if num_df.shape[1] else pd.DataFrame()

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

pair_scores = {}
for a, b in pairs:
    if a in corr.columns and b in corr.columns:
        pair_scores[(a, b)] = corr.loc[a, b]

best_pair = None
if pair_scores:
    best_pair = max(pair_scores.items(), key=lambda kv: abs(kv[1]))[0]

print("Q2) correlações consideradas:", {k: round(v, 3) for k, v in pair_scores.items()})
print("    maior |corr|:", best_pair)

Q2) correlações consideradas: {('interaction_count', 'lead_score'): np.float64(0.01), ('number_of_courses_viewed', 'lead_score'): np.float64(-0.005), ('number_of_courses_viewed', 'interaction_count'): np.float64(-0.024), ('annual_income', 'interaction_count'): np.float64(0.027)}
    maior |corr|: ('annual_income', 'interaction_count')


In [14]:
from sklearn.model_selection import train_test_split

In [15]:
# === 5) Split 60/20/20 (seed=42), sem a coluna y no X ===
# 1º: 60/40; 2º: dividir 40% em 20/20
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y if y.nunique() == 2 else None
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp if y_temp.nunique() == 2 else None
)

print(
    "Shapes ->",
    "train:", X_train.shape, 
    "| val:", X_val.shape, 
    "| test:", X_test.shape
)

Shapes -> train: (877, 8) | val: (292, 8) | test: (293, 8)


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import mutual_info_score

In [17]:
# === 6) Helper: pré-processador (One-Hot para categóricas) ===
def build_preprocessor(X_df):
    cat_cols = X_df.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
    # Compatibilidade com versões diferentes do sklearn
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    ct = ColumnTransformer(
        transformers=[
            ("cat", ohe, cat_cols),
            ("num", "passthrough", num_cols),
        ]
    )
    return ct, cat_cols, num_cols


Q3

In [18]:
# === 7) Q3 — Mutual Information de y vs. variáveis categóricas (apenas treino) ===
cat_cols_train = X_train.select_dtypes(include=["object"]).columns.tolist()
mi_scores_all = {}
for c in cat_cols_train:
    mi_scores_all[c] = round(mutual_info_score(y_train, X_train[c]), 2)

candidates_q3 = ["industry", "location", "lead_source", "employment_status"]
mi_candidates = {k: v for k, v in mi_scores_all.items() if k in candidates_q3}
best_mi_var = max(mi_candidates.items(), key=lambda kv: kv[1])[0] if mi_candidates else None

print("Q3) MI (apenas candidatas):", mi_candidates)
print("    maior MI:", best_mi_var)


Q3) MI (apenas candidatas): {'lead_source': 0.03, 'industry': 0.01, 'employment_status': 0.01, 'location': 0.0}
    maior MI: lead_source


Q4

In [19]:
# === 8) Q4 — Logistic Regression (liblinear, C=1.0, max_iter=1000, random_state=42) ===
ct, cat_cols_used, num_cols_used = build_preprocessor(X_train)
logreg = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)

pipe = Pipeline([("prep", ct), ("logreg", logreg)])
pipe.fit(X_train, y_train)
acc_val = accuracy_score(y_val, pipe.predict(X_val))
acc_val_rounded = round(acc_val, 2)

choices_q4 = [0.64, 0.74, 0.84, 0.94]
q4_choice = min(choices_q4, key=lambda x: abs(x - acc_val_rounded))

print(f"Q4) acc(val) = {acc_val:.4f} -> arredondado: {acc_val_rounded:.2f} | opção:", q4_choice)


Q4) acc(val) = 0.6815 -> arredondado: 0.68 | opção: 0.64


Q5

In [20]:
# === 9) Q5 — Eliminação de feature (diferença = acc_base - acc_sem_feature) ===
features_to_test = ["industry", "employment_status", "lead_score"]
diffs = {}

for f in features_to_test:
    cols = [c for c in X_train.columns if c != f]
    Xtr = X_train[cols]
    Xva = X_val[cols]
    ct2, _, _ = build_preprocessor(Xtr)
    logreg2 = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
    pipe2 = Pipeline([("prep", ct2), ("logreg", logreg2)])
    pipe2.fit(Xtr, y_train)
    acc2 = accuracy_score(y_val, pipe2.predict(Xva))
    diffs[f] = acc_val - acc2  # pode ser negativo

smallest_feature = min(diffs.items(), key=lambda kv: kv[1])[0]

print("Q5) diferenças (acc_base - acc_sem_feature):", {k: round(v, 5) for k, v in diffs.items()})
print("    menor diferença:", repr(smallest_feature))


Q5) diferenças (acc_base - acc_sem_feature): {'industry': -0.00685, 'employment_status': 0.0, 'lead_score': 0.00685}
    menor diferença: 'industry'


Q6

In [21]:
# === 10) Q6 — Regularização: testar C em [0.01, 0.1, 1, 10, 100] ===
Cs = [0.01, 0.1, 1, 10, 100]
accs_by_C = {}

for C in Cs:
    ctC, _, _ = build_preprocessor(X_train)
    logregC = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    pipeC = Pipeline([("prep", ctC), ("logreg", logregC)])
    pipeC.fit(X_train, y_train)
    a = accuracy_score(y_val, pipeC.predict(X_val))
    accs_by_C[C] = round(a, 3)

# Em caso de empate, escolher o menor C (opcional e reprodutível)
max_acc = max(accs_by_C.values())
best_Cs = [C for C, a in accs_by_C.items() if a == max_acc]
best_C = min(best_Cs)

print("Q6) acc(val) por C:", accs_by_C)
print("    melhor C:", best_C)


Q6) acc(val) por C: {0.01: 0.688, 0.1: 0.682, 1: 0.682, 10: 0.682, 100: 0.682}
    melhor C: 0.01
